# Dataset Pre-Processing

## Summary

The Raw Dataset, which has random dimension and duration, needs to be cleaned and such so the model builder can focus on detecting the pose movement only. This Notebook is used to do the pre-processing part of the research so all of the raw video dataset can be formated as a pose matrices file (.npy).

Each step will have a csv log (except step 1) as a record, we use it in case we have to manually modified the dataset result to have a better and cleaner npy dataset.

**BASE_FOLDER**: `../../_dataset/raw_dataset/violence_detection_dataset`

**LOG_FOLDER**: `BASE_FOLDER/logs`

## Steps

The steps included in this notebook are shown in this list:

1. **Recaling**
   
   **Input Folder**: `BASE_FOLDER/dataset_raw`
   
   **Output Folder**: `BASE_FOLDER/dataset_upscale`

   We're using FFMPEG with lanczos flag to rescale the size of each video into 640 x 640 so it will fit with YOLO-pose trained resolution. We also change the video framerate into 30 FPS as normalization.

2. **Split Scene**
   
   **Input Folder**: `BASE_FOLDER/dataset_upscale`
   
   **Output Folder**: `BASE_FOLDER/dataset_scene_split`

   **Log Data**: `LOG_FOLDER/scene_split_log.csv`

   Using [scenedetect](https://www.scenedetect.com/) python library to automatically detect any camera change. Since camera change are going to mess with the pose position, we're going to split it as different scene. This steps will also help us to create more dataset.

3. **Human Detection**
   
   **Input Folder**: `BASE_FOLDER/dataset_scene_split`
   
   **Output Folder**: `BASE_FOLDER/dataset_scene_split_violence`

   **Log Data**: `LOG_FOLDER/violence_filter_log.csv`

   YOLOv8n-pose will be used to detect human pose in the clip. We're using YOLO since it could detect multiple human in an image. YOLO pose will return the joint coordinate in 17 points, YOLO track then will make sure that we will detect the same person movement between frame.

   We approximate the potential interaction using a `PROXIMITY_TRESHOLD`, basically if the distance between two or more detected human's boundary box is less than `PROXIMITY_TRESHOLD` then we will consider it as interaction, we can then clean up each clip manually to make sure that only violence action is on the dataset.

   Each detected interaction will have one second (30 frame) padding at the start and the end of the interaction. Since each of the new clip then will only have 100 frames, a long interaction and multiple interaction will be splitted again.

4. **Pose Extraction**

   **Input Folder**: `BASE_FOLDER/dataset_scene_split`
   
   **Output Folder**: `BASE_FOLDER/dataset_npy`

   Each pose in the clip will then going to be extracted as `.npy` files, this will help the model builder to work faster. We will only going to detect any data that predicted to have violence in it.

5. **Video Preview**
   
   **Input Folder**: `BASE_FOLDER/dataset_scene_split` and `BASE_FOLDER/dataset_npy`
   
   **Output Folder**: `BASE_FOLDER/dataset_preview`

   We'll show each clip with the skeleton on top of it so we can do manual check whether the skeleton pose is accurate or not.



# Dependencies Import

In [14]:
import os
import cv2
import csv
import numpy as np
import pandas as pd
import subprocess
import shutil
from tqdm import tqdm
from scenedetect import detect, ContentDetector, split_video_ffmpeg
from ultralytics import YOLO

# Configuration

In [15]:
CLASSES = ['assault', 'fighting', 'shooting', 'robbery', 'normal_event']

DATASET_DIR = '../../_dataset/raw_dataset/violence_detection_dataset'
LOGS_DIR = f"{DATASET_DIR}/preprocess_logs"

DATASET_RAW_DIR = f"{DATASET_DIR}/violence_detection_dataset_manual_filter"
UPSCALE_DIR = f"{DATASET_DIR}/violence_detection_dataset_upscale"
SCENE_DIR = f"{DATASET_DIR}/violence_detection_dataset_scene_split"

VIOLENCE_DIR = f"{DATASET_DIR}/violence_detection_dataset_scene_split_violence"
VID_CLEAN_DIR = f"{DATASET_DIR}/violence_detection_dataset_scene_split_violence_clean"
VID_STATIC_DIR = f"{DATASET_DIR}/violence_detection_dataset_scene_split_violence_static"

NPY_DIR = f"{DATASET_DIR}/violence_detection_dataset_scene_npy"
NPY_CLEAN_DIR = f"{DATASET_DIR}/violence_detection_dataset_scene_npy_clean"   
NPY_STATIC_DIR = f"{DATASET_DIR}/violence_detection_dataset_scene_npy_static"

PREVIEW_DIR = f"{DATASET_DIR}/violence_detection_dataset_preview"

MATRIX_TARGET_FRAME = 100 # Target frame
MATRIX_MAX_PERSON = 3   # Max person
MATRIX_KEYPOINTS = 17  # Jumlah Keypoint COCO YOLO
MATRIX_KEYPOINT_CONFIDENCE = 3   # X, Y, Confidence

PROXIMITY_THRESHOLD = 150

In [19]:
for dir_path in [LOGS_DIR, UPSCALE_DIR, SCENE_DIR, VIOLENCE_DIR, VID_CLEAN_DIR, VID_STATIC_DIR, NPY_DIR, NPY_CLEAN_DIR, NPY_STATIC_DIR, PREVIEW_DIR]:
    for cls in CLASSES:
        os.makedirs(os.path.join(dir_path, cls), exist_ok=True)

print("Setup direktori selesai.")

Setup direktori selesai.


In [3]:
model = YOLO('yolov8n-pose.pt')

# Step 1 - Rescaling

In [5]:
def process_upscale():
    print("Memulai Tahap 1: Upscaling & FPS Normalization...")
    
    for cls in CLASSES:
        folder_path = os.path.join(DATASET_RAW_DIR, cls)
        if not os.path.exists(folder_path): continue
        
        videos = [v for v in os.listdir(folder_path) if v.endswith(('.mp4', '.avi', '.mkv'))]
        for vid in tqdm(videos, desc=f"Memproses {cls}"):
            input_path = os.path.join(folder_path, vid)
            output_path = os.path.join(UPSCALE_DIR, cls, vid)
            
            # Lewati jika sudah ada (berguna jika proses terhenti di tengah jalan)
            if os.path.exists(output_path): continue
            
            # Command FFmpeg: 30 FPS, resolusi 640x640 dengan Lanczos
            cmd = [
                'ffmpeg', '-y', '-hwaccel', 'auto', '-i', input_path, 
                '-r', '30', 
                '-vf', 'scale=640:640:flags=lanczos', 
                '-c:a', 'copy', # Copy audio as is
                output_path
            ]
            
            # Jalankan FFmpeg tanpa print log yang memenuhi layar
            subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [6]:
process_upscale()

Memulai Tahap 1: Upscaling & FPS Normalization...


Memproses normal_event: 100%|██████████| 211/211 [23:26<00:00,  6.67s/it] 


# Step 2 - Split Scene

In [7]:
def process_scene_split():
    print("Memulai Tahap 2: Scene Splitting...")
    
    csv_file = f'{LOGS_DIR}/scene_split_log.csv'
    
    with open(csv_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['filepath', 'has_multiple_scene', 'new_scenes'])
        
        for cls in CLASSES:
            folder_path = os.path.join(UPSCALE_DIR, cls)
            if not os.path.exists(folder_path): continue
            
            videos = [v for v in os.listdir(folder_path) if v.endswith(('.mp4', '.avi'))]
            for vid in tqdm(videos, desc=f"Mendeteksi Scene {cls}"):
                input_path = os.path.join(folder_path, vid)
                base_name = os.path.splitext(vid)[0]
                ext = os.path.splitext(vid)[1]
                
                # Deteksi Scene
                scene_list = detect(input_path, ContentDetector())
                
                has_multiple = len(scene_list) > 1
                new_scenes_paths = []
                
                if not has_multiple:
                    # Copy langsung jika tidak ada scene baru
                    out_path = os.path.join(SCENE_DIR, cls, f"{base_name}_scene_0{ext}")
                    # Gunakan copy file sederhana
                    import shutil
                    shutil.copy2(input_path, out_path)
                    new_scenes_paths.append(out_path)
                else:
                    # Split menggunakan ffmpeg internal dari scenedetect
                    for i, scene in enumerate(scene_list):
                        out_path = os.path.join(SCENE_DIR, cls, f"{base_name}_scene_{i}{ext}")
                        new_scenes_paths.append(out_path)
                    
                    # Output template
                    output_template = os.path.join(SCENE_DIR, cls, f"{base_name}_scene_$SCENE_NUMBER{ext}")
                    split_video_ffmpeg(input_path, scene_list, output_file_template=output_template, show_progress=False)
                
                writer.writerow([f"{cls}/{vid}", has_multiple, str(new_scenes_paths)])

In [8]:
process_scene_split()

Memulai Tahap 2: Scene Splitting...


Mendeteksi Scene normal_event: 100%|██████████| 211/211 [09:57<00:00,  2.83s/it]


# Step 3 - Human Detection

In [4]:
def process_violence_filter_resume():
    print("Memulai Tahap 3: Violence Filter (Dengan Fitur Resume)...")
    csv_file = csv_file = f'{LOGS_DIR}/violence_filter_log.csv'
    processed_files = set()
    
    # 1. Baca log yang sudah ada untuk mencari tahu file mana yang sudah selesai
    if os.path.exists(csv_file):
        print(f"Ditemukan file log sebelumnya. Membaca daftar file yang sudah diproses...")
        with open(csv_file, mode='r', newline='') as f:
            reader = csv.reader(f)
            header = next(reader, None) # Lewati header
            for row in reader:
                if row and len(row) > 0:
                    processed_files.add(row[0]) # Kolom pertama adalah filepath
        print(f"Total file yang sudah selesai sebelumnya: {len(processed_files)}")
    
    # 2. Buka file dengan mode 'a' (append) jika sudah ada, atau 'w' (write) jika baru
    mode = 'a' if os.path.exists(csv_file) else 'w'
    
    with open(csv_file, mode=mode, newline='') as f:
        writer = csv.writer(f)
        
        # Tulis header hanya jika file baru dibuat
        if mode == 'w':
            writer.writerow(['filepath', 'has_human', 'human_amount', 'has_violence', 'violence_frame_start', 'violence_frame_end', 'new_scenes'])
        
        for cls in CLASSES:
            folder_path = os.path.join(SCENE_DIR, cls)
            if not os.path.exists(folder_path): continue
            
            videos = [v for v in os.listdir(folder_path) if v.endswith(('.mp4', '.avi'))]
            for vid in tqdm(videos, desc=f"Menganalisis Aksi {cls}"):
                filepath_key = f"{cls}/{vid}"
                
                # --- LOGIKA RESUME ---
                if filepath_key in processed_files:
                    # Jika video sudah ada di log, lewati langsung ke video berikutnya
                    continue 
                
                input_path = os.path.join(folder_path, vid)
                cap = cv2.VideoCapture(input_path)
                total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = cap.get(cv2.CAP_PROP_FPS)
                
                has_human = False
                max_humans = 0
                violence_starts = []
                violence_ends = []
                
                frame_idx = 0
                is_violence_active = False
                current_start = 0
                
                PROXIMITY_THRESHOLD = 150 
                
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret: break
                    
                    results = model.track(frame, persist=True, classes=[0], verbose=False)
                    
                    num_humans = 0
                    if results[0].boxes is not None and results[0].boxes.id is not None:
                        boxes = results[0].boxes.xyxy.cpu().numpy()
                        num_humans = len(boxes)
                        has_human = True
                        max_humans = max(max_humans, num_humans)
                        
                        if num_humans >= 2:
                            for i in range(num_humans):
                                for j in range(i+1, num_humans):
                                    c1 = ((boxes[i][0] + boxes[i][2])/2, (boxes[i][1] + boxes[i][3])/2)
                                    c2 = ((boxes[j][0] + boxes[j][2])/2, (boxes[j][1] + boxes[j][3])/2)
                                    dist = np.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)
                                    
                                    if dist < PROXIMITY_THRESHOLD:
                                        if not is_violence_active:
                                            is_violence_active = True
                                            current_start = max(0, frame_idx - 30)
                                        break
                                        
                        elif num_humans < 2 and is_violence_active:
                            is_violence_active = False
                            violence_starts.append(current_start)
                            violence_ends.append(min(total_frames, frame_idx + 30))
                            
                    frame_idx += 1
                
                if is_violence_active:
                    violence_starts.append(current_start)
                    violence_ends.append(min(total_frames, frame_idx + 30))
                
                cap.release()
                
                new_scenes_paths = []
                for start, end in zip(violence_starts, violence_ends):
                    duration = end - start
                    start_chunks = []
                    
                    if duration > 100:
                        step = 50
                        for s in range(start, end - 100 + 1, step):
                            start_chunks.append((s, s + 100))
                    else:
                        center = (start + end) // 2
                        new_start = max(0, center - 50)
                        new_end = min(total_frames, new_start + 100)
                        start_chunks.append((new_start, new_end))
                    
                    for idx, (s_frame, e_frame) in enumerate(start_chunks):
                        out_name = f"{os.path.splitext(vid)[0]}_v{start}_{idx}.mp4"
                        out_path = os.path.join(VIOLENCE_DIR, cls, out_name)
                        
                        s_time = s_frame / fps
                        dur_time = (e_frame - s_frame) / fps
                        
                        cmd = ['ffmpeg', '-y', '-i', input_path, '-ss', str(s_time), '-t', str(dur_time), '-c:v', 'libx264', out_path]
                        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                        new_scenes_paths.append(out_path)

                # Tulis hasil video ini dan LANGSUNG flush agar tersimpan ke disk
                writer.writerow([filepath_key, has_human, max_humans, len(violence_starts)>0, str(violence_starts), str(violence_ends), str(new_scenes_paths)])
                f.flush() # MEMASTIKAN DATA LANGSUNG DITULIS KE HARDDISK (Pencegahan jika restart terjadi lagi)

In [5]:
process_violence_filter_resume()

Memulai Tahap 3: Violence Filter (Dengan Fitur Resume)...
Ditemukan file log sebelumnya. Membaca daftar file yang sudah diproses...
Total file yang sudah selesai sebelumnya: 454


Menganalisis Aksi shooting:  95%|█████████▌| 127/133 [22:53<05:13, 52.26s/it] 

Menganalisis Aksi shooting:  98%|█████████▊| 130/133 [29:11<04:40, 93.54s/it] 

Menganalisis Aksi robbery:   1%|          | 2/270 [01:23<2:33:32, 34.38s/it]

Menganalisis Aksi robbery:  41%|████      | 110/270 [6:00:26<12:12:16, 274.60s/it]

...

Menganalisis Aksi robbery:  99%|█████████▊| 266/270 [11:17:08<11:17, 169.48s/it]

Menganalisis Aksi normal_event:  69%|██████▊   | 164/239 [3:56:22<4:39:24, 223.52s/it]  

Menganalisis Aksi normal_event:  75%|███████▌  | 180/239 [5:02:27<1:35:45, 97.37s/it] 

Menganalisis Aksi normal_event:  79%|███████▊  | 188/239 [5:55:20<7:40:34, 541.86s/it]

Menganalisis Aksi normal_event: 100%|██████████| 239/239 [20:18:59<00:00, 306.02s/it]     


# Step 4 - Pose Extraction

In [8]:
def process_npy_extraction_safe():
    print("Memulai Tahap 4: Pose Extraction (.npy) - Safe Mode...")
    
    T = MATRIX_TARGET_FRAME
    M = MATRIX_MAX_PERSON
    V = MATRIX_KEYPOINTS
    C = MATRIX_KEYPOINT_CONFIDENCE
    
    for cls in CLASSES:
        folder_path = os.path.join(VIOLENCE_DIR, cls)
        if not os.path.exists(folder_path): continue
        
        videos = [v for v in os.listdir(folder_path) if v.endswith(('.mp4', '.avi'))]
        for vid in tqdm(videos, desc=f"Ekstrak Pose {cls}"):
            input_path = os.path.join(folder_path, vid)
            cap = cv2.VideoCapture(input_path)
            
            # Inisialisasi tensor awal dengan nol
            pose_data = np.zeros((C, T, V, M))
            
            frame_idx = 0
            while cap.isOpened() and frame_idx < T:
                ret, frame = cap.read()
                if not ret: break
                
                results = model.track(frame, persist=True, classes=[0], verbose=False)
                
                if results[0].keypoints is not None:
                    # Ambil data keypoint
                    kpts = results[0].keypoints.data.cpu().numpy()
                    
                    # SAFETY CHECK 1: Pastikan kpts tidak sepenuhnya kosong
                    if kpts.size > 0 and kpts.ndim == 3:
                        
                        num_people = min(len(kpts), M)
                        for m in range(num_people):
                            person_kpts = kpts[m] # Ekspektasi Shape: (17, 3)
                            
                            # SAFETY CHECK 2: Pastikan pose berhasil diekstrak (punya minimal 17 titik)
                            if len(person_kpts) < 17:
                                continue # Lewati orang ini, biarkan nilainya 0 di pose_data
                            
                            # Normalisasi Pelvis (Titik Tengah Hip: Joint 11 dan 12)
                            hip_left = person_kpts[11]
                            hip_right = person_kpts[12]
                            
                            # Jika hip terdeteksi (conf > 0)
                            if hip_left[2] > 0 and hip_right[2] > 0:
                                pelvis_x = (hip_left[0] + hip_right[0]) / 2
                                pelvis_y = (hip_left[1] + hip_right[1]) / 2
                                
                                # Kurangi x dan y dengan pelvis (conf tetap)
                                for v in range(V):
                                    if person_kpts[v][2] > 0: # Hanya jika sendi terlihat
                                        pose_data[0, frame_idx, v, m] = person_kpts[v][0] - pelvis_x # X
                                        pose_data[1, frame_idx, v, m] = person_kpts[v][1] - pelvis_y # Y
                                        pose_data[2, frame_idx, v, m] = person_kpts[v][2] # Confidence
                            else:
                                # Jika pelvis tidak ada, masukkan raw data sementara
                                for v in range(V):
                                    pose_data[0, frame_idx, v, m] = person_kpts[v][0]
                                    pose_data[1, frame_idx, v, m] = person_kpts[v][1]
                                    pose_data[2, frame_idx, v, m] = person_kpts[v][2]
                                
                frame_idx += 1
            cap.release()
            
            # --- Proses Imputasi (Interpolasi Pandas) ---
            for m in range(M):
                for v in range(V):
                    for c in range(2): # Hanya x dan y yang diinterpolasi
                        series = pd.Series(pose_data[c, :, v, m])
                        # Ganti 0 dengan NaN untuk interpolasi
                        series.replace(0.0, np.nan, inplace=True)
                        # Gunakan batasan limit_direction agar tidak error jika semuanya kosong
                        if not series.isna().all():
                            series.interpolate(method='linear', limit_direction='both', inplace=True)
                        series.fillna(0.0, inplace=True) # Jika masih kosong, kembalikan ke 0
                        pose_data[c, :, v, m] = series.to_numpy()
            
            # Simpan hasil dalam format .npy
            out_name = f"{os.path.splitext(vid)[0]}.npy"
            out_path = os.path.join(NPY_DIR, cls, out_name)
            np.save(out_path, pose_data)

In [9]:
process_npy_extraction_safe()

Memulai Tahap 4: Pose Extraction (.npy) - Safe Mode...


Ekstrak Pose assault:   0%|          | 0/2889 [00:00<?, ?it/s]

Ekstrak Pose assault:   7%|▋         | 210/2889 [29:26<6:21:06,  8.54s/it]

Ekstrak Pose assault:  25%|██▍       | 720/2889 [2:01:26<13:38:44, 22.65s/it]

Ekstrak Pose assault:  41%|████      | 1175/2889 [3:56:46<4:08:04,  8.68s/it]

...

Ekstrak Pose robbery:   1%|▏         | 47/3359 [07:02<7:51:08,  8.54s/it]

Ekstrak Pose robbery:  72%|███████▏  | 2433/3359 [7:03:22<2:07:48,  8.28s/it]

Ekstrak Pose robbery:  98%|█████████▊| 3301/3359 [9:07:43<08:26,  8.73s/it]  

Ekstrak Pose normal_event:  45%|████▌     | 2501/5552 [8:58:38<9:19:12, 11.00s/it] 

Ekstrak Pose normal_event: 100%|██████████| 5552/5552 [17:00:13<00:00, 11.03s/it]  


# Step 5 - Static Filtering

In [ ]:
def process_variance_cleaning_with_video():
    print("Memulai Tahap 6: Pembersihan Dataset & Sinkronisasi Video...")
    csv_file = f'{LOGS_DIR}/variance_cleaning_log.csv'
    
    MOTION_THRESHOLD = 10.0
    
    total_files = 0
    moved_to_clean = 0
    moved_to_static = 0
    
    with open(csv_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['npy_filename', 'is_static'])
        
        for cls in CLASSES:
            npy_source_folder = os.path.join(NPY_DIR, cls)
            vid_source_folder = os.path.join(VIOLENCE_DIR, cls)
            
            if not os.path.exists(npy_source_folder): continue
            
            npy_files = [f for f in os.listdir(npy_source_folder) if f.endswith('.npy')]
            for npy_file in tqdm(npy_files, desc=f"Membersihkan {cls}"):
                input_npy_path = os.path.join(npy_source_folder, npy_file)
                total_files += 1
                
                # 1. Menghitung Motion Score dari NPY
                pose_data = np.load(input_npy_path)
                xy_coords = pose_data[0:2, :, :, :]
                std_dev_temporal = np.std(xy_coords, axis=1) 
                motion_score = np.mean(std_dev_temporal)
                
                # Cek apakah file di bawah threshold gerak
                is_static = bool(motion_score < MOTION_THRESHOLD)
                
                # 2. Mencari Video Asli yang Berpasangan
                base_name = os.path.splitext(npy_file)[0]
                vid_file = None
                
                # Cari file dengan nama depan sama dan berekstensi video
                if os.path.exists(vid_source_folder):
                    for v in os.listdir(vid_source_folder):
                        if v.startswith(base_name) and v.endswith(('.mp4', '.avi')):
                            vid_file = v
                            break
                
                # 3. Mendistribusikan File NPY dan Video
                if not is_static:
                    # File Valid / Banyak Gerakan
                    shutil.copy2(input_npy_path, os.path.join(NPY_CLEAN_DIR, cls, npy_file))
                    if vid_file:
                        shutil.copy2(os.path.join(vid_source_folder, vid_file), os.path.join(VID_CLEAN_DIR, cls, vid_file))
                    moved_to_clean += 1
                else:
                    # File Diam / Terlalu Statis
                    shutil.copy2(input_npy_path, os.path.join(NPY_STATIC_DIR, cls, npy_file))
                    if vid_file:
                        shutil.copy2(os.path.join(vid_source_folder, vid_file), os.path.join(VID_STATIC_DIR, cls, vid_file))
                    moved_to_static += 1
                        
                # 4. Mencatat ke dalam CSV (Hanya nama file dan status boolean)
                writer.writerow([npy_file, is_static])

    print("\n=== Laporan Pembersihan Dataset ===")
    print(f"Total Klip Diproses : {total_files}")
    print(f"Klip Valid (Aktif)  : {moved_to_clean} -> NPY & Video tersimpan di '*_clean'")
    print(f"Klip Dibuang (Diam) : {moved_to_static} -> NPY & Video dipindah ke '*_static'")
    print(f"Log pembersihan disimpan pada: {csv_file}")

In [ ]:
process_variance_cleaning_with_video()

Memulai Tahap 6: Pembersihan Dataset & Sinkronisasi Video...


Membersihkan normal_event: 100%|██████████| 5552/5552 [02:09<00:00, 42.90it/s]


=== Laporan Pembersihan Dataset ===
Total Klip Diproses : 15415
Klip Valid (Aktif)  : 15280 -> NPY & Video tersimpan di '*_clean'
Klip Dibuang (Diam) : 135 -> NPY & Video dipindah ke '*_static'
Log pembersihan disimpan pada: ../../_dataset/raw_dataset/violence_detection_dataset/preprocess_logs/variance_cleaning_log.csv


: 

# Step 6 - Preview

In [10]:
EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4),          # Area Kepala / Wajah
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), # Bahu dan Lengan
    (11, 12), (5, 11), (6, 12),              # Area Torso / Badan
    (11, 13), (13, 15), (12, 14), (14, 16)   # Area Kaki
]

COLORS = [(0, 255, 255),  # Kuning untuk Orang 1
          (255, 0, 255),  # Magenta untuk Orang 2
          (0, 255, 0)]    # Hijau untuk Orang 3

In [11]:
def process_preview():
    print("Memulai Tahap 5: Membuat Video Preview Pose dari .npy...")
    
    T = MATRIX_TARGET_FRAME
    M = MATRIX_MAX_PERSON
    V = MATRIX_KEYPOINTS
    
    for cls in CLASSES:
        folder_path = os.path.join(NPY_DIR, cls)
        if not os.path.exists(folder_path): continue
        
        npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
        for npy_file in tqdm(npy_files, desc=f"Render Preview {cls}"):
            input_path = os.path.join(folder_path, npy_file)
            
            # Load matriks data dengan dimensi (3, 100, 17, 3)
            pose_data = np.load(input_path) 
            
            out_name = f"{os.path.splitext(npy_file)[0]}_preview.mp4"
            out_path = os.path.join(PREVIEW_DIR, cls, out_name)
            
            # Inisialisasi Video Writer (Kanvas 640x640, 30 FPS)
            fourcc = cv2.VideoWriter.fourcc(*'mp4v')
            out = cv2.VideoWriter(out_path, fourcc, 30.0, (640, 640))
            
            for t in range(T):
                # Buat kanvas hitam (Blank Frame)
                frame = np.zeros((640, 640, 3), dtype=np.uint8)
                
                for m in range(M):
                    color = COLORS[m]
                    
                    # Ambil array koordinat X, Y, dan Confidence untuk orang ke-m di frame ke-t
                    x_coords = pose_data[0, t, :, m]
                    y_coords = pose_data[1, t, :, m]
                    confidences = pose_data[2, t, :, m]
                    
                    # Lewati jika orang ini tidak memiliki data (semua conf 0)
                    if np.sum(confidences) == 0:
                        continue
                        
                    # 1. Menggambar Tulang (Lines)
                    for edge in EDGES:
                        pt1_idx, pt2_idx = edge
                        
                        # Gambar garis hanya jika kedua ujung sendi memiliki confidence > 0
                        if confidences[pt1_idx] > 0 and confidences[pt2_idx] > 0:
                            # Ditambah 320 untuk menggeser titik panggul (0,0) ke tengah layar 640x640
                            x1 = int(x_coords[pt1_idx] + 320)
                            y1 = int(y_coords[pt1_idx] + 320)
                            x2 = int(x_coords[pt2_idx] + 320)
                            y2 = int(y_coords[pt2_idx] + 320)
                            
                            cv2.line(frame, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
                            
                    # 2. Menggambar Titik Sendi (Joints)
                    for v in range(V):
                        if confidences[v] > 0:
                            x = int(x_coords[v] + 320)
                            y = int(y_coords[v] + 320)
                            # Beri titik putih untuk setiap joint yang valid
                            cv2.circle(frame, (x, y), 3, (255, 255, 255), -1, cv2.LINE_AA)
                
                # Tulis frame ke dalam video
                out.write(frame)
                
            out.release()

In [13]:
process_preview()

Memulai Tahap 5: Membuat Video Preview Pose dari .npy...


Render Preview normal_event: 100%|██████████| 5552/5552 [21:02<00:00,  4.40it/s]
